In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import json

CSV_PATH = "../Data/tmdb_semantic_catalog_alllangs_with_new_movies.csv"

if "CACHED_DF" not in globals():
    try:
        CACHED_DF = pd.read_csv(CSV_PATH, low_memory=False)
        print(f"Loaded dataset into memory: {len(CACHED_DF):,} rows, {len(CACHED_DF.columns)} columns.")
    except FileNotFoundError:
        raise FileNotFoundError(f"CSV not found at {CSV_PATH}. Run from CineMatch root or update CSV_PATH.")
else:
    print("Using cached dataset in memory (CACHED_DF).")

# Use the cached frame for UI operations
df = CACHED_DF

candidate_title_cols = ["title", "Title", "movie_title", "movieTitle", "name", "original_title"]
candidate_year_cols = ["year", "Year", "release_year", "releaseDate", "release_date"]

title_col = next((c for c in candidate_title_cols if c in df.columns), None)
year_col = next((c for c in candidate_year_cols if c in df.columns), None)
if title_col is None:
    raise ValueError(
        f"Could not find a title column. Available columns: {list(df.columns)}"
    )

# Widgets
search_box = widgets.Text(value="", placeholder="Type a movie title...", description="Search:", layout=widgets.Layout(width="60%"))
max_results_slider = widgets.IntSlider(value=10, min=1, max=200, step=1, description="Max rows:", continuous_update=False, layout=widgets.Layout(width="35%"))
case_checkbox = widgets.Checkbox(value=False, description="Case sensitive", indent=False)
search_btn = widgets.Button(description="Search", button_style="primary")
reset_btn = widgets.Button(description="Reset", button_style="")

# List / selection widgets
matches_select = widgets.Select(options=[], rows=8, description="Matches:", layout=widgets.Layout(width="100%"))
show_btn = widgets.Button(description="Show movie", button_style="success")
out = widgets.Output()

# Hold last matches DataFrame so selection can reference original indices
LAST_MATCHES = None


def make_display_label(row_index, row):
    title = str(row.get(title_col, "")).strip()
    year = str(row.get(year_col, "") if year_col else "") if year_col else ""
    label = f"{title}"
    if year:
        label += f" ({year})"
    # include the dataframe index so we can map back
    return f"{row_index}: {label}"


def run_search(_=None):
    global LAST_MATCHES
    with out:
        clear_output(wait=True)
        query = search_box.value.strip()
        max_results = max_results_slider.value

        if not query:
            display(HTML("<b>Enter a title to search.</b>"))
            matches_select.options = []
            return

        matches = df[df[title_col].astype(str).str.contains(query, case=case_checkbox.value, na=False)]

        if matches.empty:
            display(HTML(f"<b>No matches found for:</b> {query}"))
            matches_select.options = []
            LAST_MATCHES = None
            return

        LAST_MATCHES = matches
        display(HTML(f"<b>{len(matches)} match(es)</b>. Showing first {min(max_results, len(matches))}."))

        # Build options: display_label -> original index
        rows = matches.head(max_results)
        options = []
        for idx, r in rows.iterrows():
            options.append((make_display_label(idx, r), idx))

        matches_select.options = options


def run_show(_=None):
    with out:
        clear_output(wait=True)
        if LAST_MATCHES is None:
            display(HTML("<b>No search results to select from.</b>"))
            return
        if not matches_select.value and matches_select.value != 0:
            display(HTML("<b>No movie selected.</b>"))
            return

        selected_idx = matches_select.value
        if selected_idx not in LAST_MATCHES.index:
            display(HTML("<b>Selected movie not found in cached matches.</b>"))
            return

        row = LAST_MATCHES.loc[selected_idx]

        # Render full vertical description without truncation
        parts = ["<div style='font-family:monospace'>"]
        for col in row.index:
            val = row[col]
            if pd.isna(val):
                val_str = ""
            else:
                # If looks like JSON, pretty print
                if isinstance(val, str) and (val.strip().startswith("{") or val.strip().startswith("[")):
                    try:
                        parsed = json.loads(val)
                        val_str = json.dumps(parsed, indent=2, ensure_ascii=False)
                    except Exception:
                        val_str = val
                else:
                    val_str = str(val)

            # escape HTML in val_str
            safe_val = (val_str
                        .replace("&", "&amp;")
                        .replace("<", "&lt;")
                        .replace(">", "&gt;"))

            parts.append(f"<div style=\"margin-bottom:8px\"><b>{col}</b><div style=\"white-space:pre-wrap; margin-left:8px; color:#111\">{safe_val}</div></div>")

        parts.append("</div>")
        html = "\n".join(parts)
        display(HTML(html))


def run_reset(_=None):
    global LAST_MATCHES
    search_box.value = ""
    matches_select.options = []
    LAST_MATCHES = None
    with out:
        clear_output(wait=True)
        display(HTML("<b>Search reset.</b>"))

# Wire events
search_btn.on_click(run_search)
show_btn.on_click(run_show)
reset_btn.on_click(run_reset)
search_box.on_submit(run_search)

# Layout
controls = widgets.HBox([search_box, max_results_slider])
actions = widgets.HBox([case_checkbox, search_btn, show_btn, reset_btn])
ui = widgets.VBox([controls, actions, matches_select, out])

display(ui)


Loaded dataset into memory: 1,320,040 rows, 30 columns.


/var/folders/lq/r8qjzl2x10b2vksfq2gq9m080000gn/T/ipykernel_11558/165833728.py:151: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  search_box.on_submit(run_search)
